In [1]:
# Upgrade installer tooling and align LangChain packages for compatibility
# Run this cell, then RESTART the kernel, then re-run the environment-check cell (previous cell) and the rest.
%pip install -U pip setuptools wheel
%pip install -U "langchain>=0.0.325" langchain-huggingface langchain-community
# Reinstall torch (CPU wheel) to repair any corrupted installs; remove --no-deps if you need extras
%pip install --force-reinstall --no-deps torch --index-url https://download.pytorch.org/whl/cpu

print("Finished package upgrades. Please restart the kernel before running other cells.")

Defaulting to user installation because normal site-packages is not writeable
  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
Using cached setuptools-80.9.0-py3-none-any.whl (1.2 MB)
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.9.0+cpu requires torch==2.9.0, but you have torch 2.9.1 which is incompatible.
torchvision 0.24.0+cpu requires torch==2.9.0, but you have torch 2.9.1 which is incompatible.


Defaulting to user installation because normal site-packages is not writeable
  Using cached langchain-1.1.0-py3-none-any.whl.metadata (4.9 kB)
Using cached langchain-1.1.0-py3-none-any.whl (101 kB)
Note: you may need to restart the kernel to use updated packages.


Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cpu
   ---------------------------------------- 0.0/110.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/110.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/110.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/110.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/110.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/110.9 MB 421.9 kB/s eta 0:04:22
   ---------------------------------------- 0.5/110.9 MB 421.9 kB/s eta 0:04:22
   ---------------------------------------- 0.8/110.9 MB 492.2 kB/s eta 0:03:44
   ---------------------------------------- 0.8/110.9 MB 492.2 kB/s eta 0:03:44
   ---------------------------------------- 0.8/110.9 MB 492.2 kB/s eta 0:03:44
   ---------------------------------------- 1.0/110.9 MB 478.8 kB/s eta 0:03:50
   ---------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
# Environment check + ensure packages are installed into the notebook kernel
import sys
print('Python executable:', sys.executable)
print('Python sys.path (first 5):', sys.path[:5])

# Show whether the package is already available in this kernel
try:
    import pkg_resources
    print('pkg_resources available')
    try:
        print('langchain-huggingface:', pkg_resources.get_distribution('langchain-huggingface'))
    except Exception as e:
        print('langchain-huggingface not found via pkg_resources:', e)
except Exception as e:
    print('pkg_resources not available:', e)

# Install required packages into the kernel (this uses the same Python as the kernel)
%pip install -q langchain-huggingface langchain-community sentence-transformers faiss-cpu transformers torch pandas

# Try importing the module we need and report result
try:
    import importlib
    m = importlib.import_module('langchain_huggingface')
    print('Successfully imported langchain_huggingface:', getattr(m, '__version__', 'no __version__'))
except Exception as e:
    print('Import of langchain_huggingface failed after installation:', type(e).__name__, e)

# Also ensure langchain_community vectorstores is importable
try:
    import importlib
    m2 = importlib.import_module('langchain_community.vectorstores')
    print('Successfully imported langchain_community.vectorstores')
except Exception as e:
    print('Import of langchain_community.vectorstores failed:', type(e).__name__, e)

# Now you can run the following cell(s) after this completes. If imports still fail, please restart the kernel and re-run this cell.


Python executable: c:\ProgramData\miniconda3\python.exe
Python sys.path (first 5): ['c:\\ProgramData\\miniconda3\\python313.zip', 'c:\\ProgramData\\miniconda3\\DLLs', 'c:\\ProgramData\\miniconda3\\Lib', 'c:\\ProgramData\\miniconda3', '']
pkg_resources available
langchain-huggingface: langchain-huggingface 1.1.0


C:\Users\esraa\AppData\Local\Temp\ipykernel_7276\2437328442.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Note: you may need to restart the kernel to use updated packages.


c:\ProgramData\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Successfully imported langchain_huggingface: no __version__
Successfully imported langchain_community.vectorstores


In [4]:
%pip install pypdf

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [5]:
import time
import pandas as pd
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# --- NEW IMPORTS FOR PDF ---
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ================= CONFIGURATION =================
CSV_FILE_PATH = 'data.csv'
PDF_FILE_PATH = 'machine chatbot.pdf' # <--- Make sure this matches your PDF name
INDEX_NAME = "faiss_index_136k"
MODEL_NAME = "all-MiniLM-L6-v2"

# FOR SMALL DATA (1000 rows): Set this to 1 for maximum precision.
# The chatbot will find the EXACT row instead of a group of rows.
ROWS_PER_CHUNK = 1  
# =================================================

print("--- [Build File]: Starting the Index creation process ---")

# =================================================
# PART 1: PROCESS THE CSV (Your existing logic)
# =================================================
print(f"--- Processing CSV: {CSV_FILE_PATH} ---")
df = pd.read_csv(CSV_FILE_PATH)
csv_documents = []
current_chunk = []

for index, row in df.iterrows():
    # Convert row to string
    row_text = ", ".join([f"{col}: {val}" for col, val in row.items()])
    current_chunk.append(row_text)
    
    # Group rows
    if len(current_chunk) >= ROWS_PER_CHUNK:
        combined_text = "\n".join(current_chunk)
        csv_documents.append(Document(page_content=combined_text, metadata={"source": CSV_FILE_PATH, "type": "data_row"}))
        current_chunk = []

if current_chunk:
    combined_text = "\n".join(current_chunk)
    csv_documents.append(Document(page_content=combined_text, metadata={"source": CSV_FILE_PATH, "type": "data_row"}))

print(f"✅ CSV Processed: Reduced {len(df)} rows into {len(csv_documents)} vector documents.")

# =================================================
# PART 2: PROCESS THE PDF (New Logic)
# =================================================
print(f"\n--- Processing PDF: {PDF_FILE_PATH} ---")
pdf_documents = []

try:
    # 1. Load the PDF
    loader = PyPDFLoader(PDF_FILE_PATH)
    raw_pdf_pages = loader.load()
    
    # 2. Split the PDF text
    # Smaller chunks (500) are better for small PDFs to isolate specific definitions.
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=50,
        separators=["\n\n", "\n", ".", " ", ""]
    )
    
    pdf_documents = text_splitter.split_documents(raw_pdf_pages)
    
    # Add metadata so the chatbot knows this is a description
    for doc in pdf_documents:
        doc.metadata["source"] = PDF_FILE_PATH
        doc.metadata["type"] = "feature_description"
        
    print(f"✅ PDF Processed: Created {len(pdf_documents)} description chunks.")

except Exception as e:
    print(f"⚠️ Warning: Could not process PDF. Error: {e}")
    print("Continuing with CSV data only...")

# =================================================
# PART 3: COMBINE AND BUILD INDEX
# =================================================
print("\n--- Building Vector Store ---")

# Combine both lists
all_documents = csv_documents + pdf_documents
print(f"Total documents to embed: {len(all_documents)}")

print(f"Loading Embeddings model: {MODEL_NAME}...")
embeddings = HuggingFaceEmbeddings(
    model_name=MODEL_NAME,
    model_kwargs={'device': 'cpu'}
)

start_time = time.time()

# Create index from the combined list
vector_store = FAISS.from_documents(all_documents, embeddings)

end_time = time.time()
print(f"...Finished! Time taken: {(end_time - start_time) / 60:.2f} minutes")

# Save
print(f"Saving to folder: {INDEX_NAME}...")
vector_store.save_local(INDEX_NAME)

print(f"🎉 Success! The index now contains both your Data Rows AND your PDF Feature Descriptions.")

--- [Build File]: Starting the Index creation process ---
--- Processing CSV: data.csv ---
✅ CSV Processed: Reduced 1000 rows into 1000 vector documents.

--- Processing PDF: machine chatbot.pdf ---
✅ PDF Processed: Created 21 description chunks.

--- Building Vector Store ---
Total documents to embed: 1021
Loading Embeddings model: all-MiniLM-L6-v2...
...Finished! Time taken: 1.10 minutes
Saving to folder: faiss_index_136k...
🎉 Success! The index now contains both your Data Rows AND your PDF Feature Descriptions.
